# 1 Build Modelling Dataset v1

## 1.0 Purpose

This notebook builds the frozen modelling dataset for the email engagement prediction stage.

The goal is not to repeat the full EDA. The goal is to reconstruct one clean user-campaign level dataframe where each row represents one recipient receiving one campaign/email.

The final frozen dataset will include:
- ID and audit columns
- open and click target variables
- non-leaky candidate recipient features
- non-leaky candidate campaign/content features
- temporally valid historical engagement features

This notebook avoids using all-time outcome summaries or exploratory EDA aggregates as model features.

## 1.1 Imports and file paths

In [1]:
# imports

# core libraries
import pandas as pd
import numpy as np

# utilities
import re
from pathlib import Path

# display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
# project paths

# root project folder
PROJECT_ROOT = Path.home() / 'Documents' / 'Thesis'

# main folders
RAW_DATA_DIR = PROJECT_ROOT / 'Data' / 'Raw Data'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'Data' / 'Processed Data'
NOTEBOOK_DIR = PROJECT_ROOT / 'Notebooks'
DOCS_DIR = PROJECT_ROOT / 'Documents'
OUTPUT_DIR = PROJECT_ROOT / 'Outputs'

# create folders
RAW_DATA_DIR.mkdir(parents = True, exist_ok = True)
PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok = True)
NOTEBOOK_DIR.mkdir(parents = True, exist_ok = True)
DOCS_DIR.mkdir(parents = True, exist_ok = True)
OUTPUT_DIR.mkdir(parents = True, exist_ok = True)


In [3]:
# raw data
DATASET_1_PATH = RAW_DATA_DIR / 'Dataset 1 UVA.csv'
DATASET_2_PATH = RAW_DATA_DIR / 'Datset 2 UVA lijst mailings.xlsx'
CAMPAIGN_DATE_PATH = RAW_DATA_DIR / 'UVA Robin .xlsx'

In [4]:
print(list(PROJECT_ROOT.iterdir()))

[PosixPath('/Users/khanhhien/Documents/Thesis/Quantitative Analysis Paper'), PosixPath('/Users/khanhhien/Documents/Thesis/.DS_Store'), PosixPath('/Users/khanhhien/Documents/Thesis/BSc_BAN_Thesis_Template'), PosixPath('/Users/khanhhien/Documents/Thesis/Documents'), PosixPath('/Users/khanhhien/Documents/Thesis/BSc_BAN_Thesis_Template.zip'), PosixPath('/Users/khanhhien/Documents/Thesis/Data'), PosixPath('/Users/khanhhien/Documents/Thesis/Outputs'), PosixPath('/Users/khanhhien/Documents/Thesis/Notebooks'), PosixPath('/Users/khanhhien/Documents/Thesis/Topics'), PosixPath('/Users/khanhhien/Documents/Thesis/Clicks')]


In [5]:
# verify files exist
print('Project root:', PROJECT_ROOT)
print('Dataset 1 exists:', DATASET_1_PATH.exists())
print('Dataset 2 exists:', DATASET_2_PATH.exists())
print('Campaign date dataset exists:', CAMPAIGN_DATE_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Dataset 1 exists: True
Dataset 2 exists: True
Campaign date dataset exists: True


In [6]:
# notebook metadata
FREEZE_VERSION = 'v1'

TARGETS = [
    'open',
    'click'
]

print(f'Building modelling dataset {FREEZE_VERSION}')

Building modelling dataset v1


## 1.2 Load raw data

In [ ]:
# load raw recipient data
df_recipients_raw = pd.read_csv(DATASET_1_PATH, sep=';')

# load raw campaign data
df_campaigns_raw = pd.read_excel(DATASET_2_PATH)

# load campaign date
df_schedule_raw = pd.read_excel(CAMPAIGN_DATE_PATH)

In [8]:
print('Recipients raw shape:', df_recipients_raw.shape)
print('Campaigns raw shape:', df_campaigns_raw.shape)
print('Schedule raw shape:', df_schedule_raw.shape)

Recipients raw shape: (20000, 9)
Campaigns raw shape: (337, 4)
Schedule raw shape: (1400, 11)


In [9]:
display(df_recipients_raw.head())
display(df_campaigns_raw.head())
display(df_schedule_raw.head())

,id,geslacht,geboortedatum,postcode,woonplaats,mailings,opens,clicks,interesses
0,5cb9a38136dd1336b9c528d1,m,0-0-0,NaN,NaN,"1:668,1:692,1:714,1:733,1:761,1:785,1:804,1:82...","1:879,1:891,1:913,1:935,1:996,1:1031,1:1007,1:...",NaN,"energie,loterij,kranten,entertainment,winactie..."
1,5cb9a38336dd1336b9c537f3,v,0-0-0,NaN,NaN,"1:668,1:692,1:714,1:721,1:729,1:735,1:733,1:75...","1:668,1:692,1:714,1:733,1:753,1:777,1:770,1:75...",NaN,"auto,loterij,cx80,kranten,mkb,verzekering,ente..."
2,5cb9a38236dd1336b9c52f07,m,0-0-0,NaN,NaN,"1:668,1:692,1:714,1:733,1:761,1:785,1:804,1:82...","1:876,1:961,1:1159",NaN,"auto,kranten,Auto inruilen,Windows,bladen,goed..."
3,5cb9a38236dd1336b9c53694,v,0-0-0,NaN,NaN,"1:654,1:668,1:695,1:692,1:706,1:714,1:653,1:73...","1:654,1:706,1:733,1:831,1:866,1:876,1:891,1:91...",NaN,"cx60,auto,loterij,cx80,kranten,mkb,verzekering..."
4,5cb9a38636dd1336b9c54e48,m,0-0-0,NaN,NaN,"1:668,1:692,1:714,1:733,1:760,1:785,1:804,1:82...","1:668,1:692,1:714,1:760,1:785,1:804,1:825,1:84...",NaN,"auto,kranten,verzekering,winactie,Auto inruile..."


,ID,Mailing,Subjectline,Preheader
0,1422,2026/04/22 CENTRAAL BEHEER_RNC_2,Hoe lang kun jij zonder inkomen als je arbeids...,NaN
1,1420,2026/04/21 CENTRAAL BEHEER_RNC_1,Hoe lang kun jij zonder inkomen als je arbeids...,NaN
2,1419,2026/04/21 Direct Doen wk17 - Donald Duck Stra...,Wat zit er in Donalds strandtas?,Ontdek het pakket en lees de leukste avonturen...
3,1418,2026/04/21 CRS wk17 - Donald Duck Strandpakket,Wat zit er in Donalds strandtas?,Ontdek het pakket en lees de leukste avonturen...
4,1417,2026/04/21 CPX wk17 - Donald Duck Strandpakket,Wat zit er in Donalds strandtas?,Ontdek het pakket en lees de leukste avonturen...


,Unnamed: 0,Subjectline,Mail ID,Dag,Datum,Klant / bureau,Campagnenaam,Time stamp,openInterst / clickInterest,Mail content,Unnamed: 10
0,ntvx,NaN,1417,Donderdag,2026-04-23 00:00:00,CPX,Donald Duck + Strandpakket,-Akkoord,"bladen, cadeaux",NaN,NaN
1,NaN,Wat zit er in Donalds strandtas?,NaN,NaN,NaN,NaN,NaN,Subjectline: Wat zit er in Donalds strandtas?,NaN,https://mail.omg.nl/x/?S7Y1.J9ra2hiaPm.CMjMyU_...,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pre-header: Ontdek het pakket en lees de leuks...,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260422 - CPX - CPX Donald Duck + Strandpakket ...,NaN,NaN,NaN
4,ntvx,NaN,1418,Donderdag,2026-04-23 00:00:00,CRS,Donald Duck + Strandpakket,-Akkoord,"bladen, cadeaux",https://mail.omg.nl/x/?S7Y1.J9ra2hiaPm.CMjMyU_...,NaN


In [11]:
print('Recipient columns:')
print(df_recipients_raw.columns.tolist())

print('\nCampaign columns:')
print(df_campaigns_raw.columns.tolist())

print('\nSchedule columns:')
print(df_schedule_raw.columns.tolist())

Recipient columns:
['id', 'geslacht', 'geboortedatum', 'postcode', 'woonplaats', 'mailings', 'opens', 'clicks', 'interesses']

Campaign columns:
['ID', 'Mailing', 'Subjectline', 'Preheader']

Schedule columns:
['Unnamed: 0', 'Subjectline', 'Mail ID ', 'Dag', 'Datum', 'Klant / bureau', 'Campagnenaam', 'Time stamp ', 'openInterst / clickInterest ', 'Mail content ', 'Unnamed: 10']


## 1.3 Clean recipient-level data

In [12]:
df_recipients = df_recipients_raw.copy()

In [13]:
# rename recipient columns
df_recipients = df_recipients.rename(columns={
    'id': 'user_id',
    'geslacht': 'gender',
    'geboortedatum': 'birth_date',
    'woonplaats': 'city',
    'interesses': 'interests'
})

In [14]:
# clean gender
df_recipients['gender'] = (
    df_recipients['gender']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'v': 'female',
        'mevrouw': 'female',
        'm': 'male',
        'de heer': 'male',
        'o': 'other',
        'nan': np.nan
    })
)

In [15]:
# clean birth date and create age
df_recipients['birth_date'] = (
    df_recipients['birth_date']
    .replace(['0', '0-0-0', '0000-00-00'], pd.NA)
)

df_recipients['birth_date'] = pd.to_datetime(
    df_recipients['birth_date'],
    errors = 'coerce'
)

REFERENCE_DATE = pd.Timestamp('2025-01-01')

df_recipients['age'] = (
    (REFERENCE_DATE - df_recipients['birth_date']).dt.days // 365
)

df_recipients['age_clean'] = df_recipients['age'].where(
    (df_recipients['age'] >= 18) &
    (df_recipients['age'] <= 90)
)

df_recipients['age_group'] = pd.cut(
    df_recipients['age_clean'],
    bins=[18, 35, 50, 65, 90],
    labels=['18-35', '35-50', '50-65', '65+'],
    include_lowest = True
)

In [16]:
# clean interests into list format
df_recipients['interests'] = (
    df_recipients['interests']
    .fillna('')
    .astype(str)
    .str.split(',')
)

df_recipients['interests'] = df_recipients['interests'].apply(
    lambda items: [i.strip().lower() for i in items if i.strip() != '']
)

df_recipients['n_interests'] = df_recipients['interests'].apply(len)

In [17]:
# basic check
print("Recipient cleaned shape:", df_recipients.shape)

display(df_recipients[[
    'user_id', 'gender', 'birth_date', 'age', 'age_clean',
    'age_group', 'interests', 'n_interests'
]].head())

Recipient cleaned shape: (20000, 13)


,user_id,gender,birth_date,age,age_clean,age_group,interests,n_interests
0,5cb9a38136dd1336b9c528d1,male,NaT,NaN,NaN,NaN,"[energie, loterij, kranten, entertainment, win...",13
1,5cb9a38336dd1336b9c537f3,female,NaT,NaN,NaN,NaN,"[auto, loterij, cx80, kranten, mkb, verzekerin...",30
2,5cb9a38236dd1336b9c52f07,male,NaT,NaN,NaN,NaN,"[auto, kranten, auto inruilen, windows, bladen...",13
3,5cb9a38236dd1336b9c53694,female,NaT,NaN,NaN,NaN,"[cx60, auto, loterij, cx80, kranten, mkb, verz...",35
4,5cb9a38636dd1336b9c54e48,male,NaT,NaN,NaN,NaN,"[auto, kranten, verzekering, winactie, auto in...",25


In [19]:
# recipient cleaning validation checks

print('Raw recipients:', df_recipients_raw.shape)
print('Cleaned recipients:', df_recipients.shape)

print('\nGender distribution:')
display(df_recipients['gender'].value_counts(dropna = False))

print('\nAge summary:')
display(df_recipients['age_clean'].describe())

print('\nAge group distribution:')
display(df_recipients['age_group'].value_counts(dropna = False))

print('\nInterest count summary:')
display(df_recipients['n_interests'].describe())

# basic safety checks
assert df_recipients.shape[0] == df_recipients_raw.shape[0], 'Row count changed during recipient cleaning'
assert 'user_id' in df_recipients.columns, 'user_id column missing'
assert 'gender' in df_recipients.columns, 'gender column missing'
assert 'age_group' in df_recipients.columns, 'age_group column missing'
assert 'interests' in df_recipients.columns, 'interests column missing'

print('\nRecipient cleaning checks passed')

Raw recipients: (20000, 9)
Cleaned recipients: (20000, 13)

Gender distribution:


gender
female    11075
male       8516
other       409
Name: count, dtype: int64


Age summary:


count    4836.000000
mean       50.915219
std        14.626570
min        21.000000
25%        38.000000
50%        51.000000
75%        63.000000
max        90.000000
Name: age_clean, dtype: float64


Age group distribution:


age_group
NaN      15164
35-50     1486
50-65     1468
65+        966
18-35      916
Name: count, dtype: int64


Interest count summary:


count    20000.000000
mean        19.961250
std         10.210919
min          0.000000
25%         11.000000
50%         21.000000
75%         28.000000
max         49.000000
Name: n_interests, dtype: float64


Recipient cleaning checks passed


## 1.4 Parse mailing, open, and click ID lists

In [20]:
def classify_id_list_value(x):
    """
    Classify whether a raw mailing/open/click field is missing,
    contains valid campaign IDs, or contains non-standard content.
    """
    if pd.isna(x) or str(x).strip() == "":
        return "missing"
    
    x = str(x).strip()
    
    if re.search(r"\b1:\d{3,4}\b", x):
        return "id_format"
    
    return "non_id_format"


def extract_campaign_ids(text):
    """
    Extract numeric campaign IDs from strings such as:
    '1:1234, 1:5678'
    
    Returns a list of integers.
    """
    if pd.isna(text) or str(text).strip() == "":
        return []
    
    ids = []
    
    for item in str(text).split(","):
        item = item.strip()
        
        if ":" in item:
            try:
                ids.append(int(item.split(":")[1]))
            except ValueError:
                pass
    
    return ids

In [21]:
for col in ["mailings", "opens", "clicks"]:
    type_col = f"{col[:-1]}_type" if col.endswith("s") else f"{col}_type"
    df_recipients[type_col] = df_recipients[col].apply(classify_id_list_value)

In [22]:
df_recipients["mailing_ids"] = df_recipients["mailings"].apply(extract_campaign_ids)

df_recipients["open_ids"] = df_recipients.apply(
    lambda row: extract_campaign_ids(row["opens"]) 
    if row["open_type"] == "id_format" else [],
    axis=1
)

df_recipients["click_ids"] = df_recipients.apply(
    lambda row: extract_campaign_ids(row["clicks"]) 
    if row["click_type"] == "id_format" else [],
    axis=1
)

In [23]:
print("Mailing type distribution:")
display(df_recipients["mailing_type"].value_counts(dropna=False))

print("\nOpen type distribution:")
display(df_recipients["open_type"].value_counts(dropna=False))

print("\nClick type distribution:")
display(df_recipients["click_type"].value_counts(dropna=False))

print("\nMailing ID count per user:")
display(df_recipients["mailing_ids"].apply(len).describe())

print("\nOpen ID count per user:")
display(df_recipients["open_ids"].apply(len).describe())

print("\nClick ID count per user:")
display(df_recipients["click_ids"].apply(len).describe())

# Safety checks
assert "mailing_ids" in df_recipients.columns
assert "open_ids" in df_recipients.columns
assert "click_ids" in df_recipients.columns

assert df_recipients["mailing_ids"].apply(lambda x: isinstance(x, list)).all()
assert df_recipients["open_ids"].apply(lambda x: isinstance(x, list)).all()
assert df_recipients["click_ids"].apply(lambda x: isinstance(x, list)).all()

print("\nParsing checks passed.")

Mailing type distribution:


mailing_type
id_format    20000
Name: count, dtype: int64


Open type distribution:


open_type
id_format        17793
non_id_format     2037
missing            170
Name: count, dtype: int64


Click type distribution:


...


Open ID count per user:


count    20000.000000
mean        28.351700
std         22.072996
min          0.000000
25%          8.000000
50%         26.000000
75%         46.000000
max         98.000000
Name: open_ids, dtype: float64


Click ID count per user:


count    20000.000000
mean         0.059750
std          0.786836
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         45.000000
Name: click_ids, dtype: float64


Parsing checks passed.


In [24]:
print(df_recipients["mailing_ids"].apply(len).describe())

count    20000.000000
mean        57.815200
std         21.067557
min          8.000000
25%         47.000000
50%         60.000000
75%         72.000000
max        120.000000
Name: mailing_ids, dtype: float64


## 1.5 Explode to user-campaign observations - Create open and clicks outcomes

### 1.5.1 Explode to user-campaign observations

In [25]:
# explode mailing ids
df_exploded = df_recipients.explode("mailing_ids").copy()

df_exploded = df_exploded.rename(columns={
    "mailing_ids": "mailing_id"
})

In [26]:
# clean mailing id
df_exploded["mailing_id"] = pd.to_numeric(
    df_exploded["mailing_id"],
    errors="coerce"
).astype("Int64")

In [27]:
# basic checks
print("Recipients shape:", df_recipients.shape)
print("Exploded shape:", df_exploded.shape)

print("\nUnique users:", df_exploded["user_id"].nunique())
print("Unique campaigns:", df_exploded["mailing_id"].nunique())

print("\nMissing mailing_id rows:", df_exploded["mailing_id"].isna().sum())

display(df_exploded[[
    "user_id", "mailing_id", "gender", "age_group",
    "mailing_type", "open_type", "click_type",
    "open_ids", "click_ids"
]].head())

Recipients shape: (20000, 19)
Exploded shape: (1156304, 19)

Unique users: 20000
Unique campaigns: 274

Missing mailing_id rows: 0


,user_id,mailing_id,gender,age_group,mailing_type,open_type,click_type,open_ids,click_ids
0,5cb9a38136dd1336b9c528d1,668,male,NaN,id_format,id_format,missing,"[879, 891, 913, 935, 996, 1031, 1007, 1013, 10...",[]
0,5cb9a38136dd1336b9c528d1,692,male,NaN,id_format,id_format,missing,"[879, 891, 913, 935, 996, 1031, 1007, 1013, 10...",[]
0,5cb9a38136dd1336b9c528d1,714,male,NaN,id_format,id_format,missing,"[879, 891, 913, 935, 996, 1031, 1007, 1013, 10...",[]
0,5cb9a38136dd1336b9c528d1,733,male,NaN,id_format,id_format,missing,"[879, 891, 913, 935, 996, 1031, 1007, 1013, 10...",[]
0,5cb9a38136dd1336b9c528d1,761,male,NaN,id_format,id_format,missing,"[879, 891, 913, 935, 996, 1031, 1007, 1013, 10...",[]


In [28]:
# safety checks
expected_rows = df_recipients["mailing_ids"].apply(len).sum()
actual_rows = len(df_exploded)

print("Expected exploded rows:", expected_rows)
print("Actual exploded rows:", actual_rows)

assert actual_rows == expected_rows, "Exploded row count does not match total mailing IDs."
assert df_exploded["mailing_id"].notna().all(), "Some exploded rows have missing mailing_id."

print("\nExplode checks passed.")

Expected exploded rows: 1156304
Actual exploded rows: 1156304

Explode checks passed.


In [29]:
print(df_exploded["age_group"].value_counts(dropna=False))

age_group
NaN      859788
50-65     94445
35-50     88108
65+       63309
18-35     50654
Name: count, dtype: int64


### 1.5.2 Create open and clicks outcomes

In [30]:
def check_open(row):
    """
    Create open outcome for one user-campaign row.
    """
    if row["open_type"] == "missing":
        return 0
    
    if row["open_type"] != "id_format":
        return np.nan
    
    return 1 if row["mailing_id"] in row["open_ids"] else 0


def check_click(row):
    """
    Create click outcome for one user-campaign row.
    """
    if row["click_type"] == "missing":
        return 0
    
    if row["click_type"] != "id_format":
        return np.nan
    
    return 1 if row["mailing_id"] in row["click_ids"] else 0

In [31]:
df_exploded["open"] = df_exploded.apply(check_open, axis=1)
df_exploded["click"] = df_exploded.apply(check_click, axis=1)

In [ ]:
print("Open value counts:")
display(df_exploded["open"].value_counts(dropna=False))

print("\nClick value counts:")
display(df_exploded["click"].value_counts(dropna=False))

print("\nOpen rate:", df_exploded["open"].mean())
print("Click rate:", df_exploded["click"].mean())

print("\nOpen missing rate:", df_exploded["open"].isna().mean())
print("Click missing rate:", df_exploded["click"].isna().mean())

Open value counts:


open
1.0    567034
0.0    490510
NaN     98760
Name: count, dtype: int64


Click value counts:


click
0.0    1103047
NaN      52062
1.0       1195
Name: count, dtype: int64


Open rate: 0.5361800549197008
Click rate: 0.001082190316977619

Open missing rate: 0.08541006517317246
Click missing rate: 0.045024491829138355


## 1.6 Merge campaign metadata

In [ ]:
# rename campaign columns
df_campaigns = df_campaigns_raw.copy()

df_campaigns = df_campaigns.rename(columns={
    "ID": "mailing_id",
    "Mailing": "mailing_info",
    "Subjectline": "subject_line",
    "Preheader": "preheader"
})

In [34]:
# keep only necessary columns
campaign_cols = [
    "mailing_id",
    "mailing_info",
    "subject_line",
    "preheader"
]

df_campaigns = df_campaigns[campaign_cols].copy()

In [35]:
# clean mailing id
df_campaigns["mailing_id"] = pd.to_numeric(
    df_campaigns["mailing_id"],
    errors="coerce"
).astype("Int64")

In [36]:
# remove duplicate campaign rows
print("Campaign rows before dropping duplicates:", df_campaigns.shape)

df_campaigns = df_campaigns.drop_duplicates(subset=["mailing_id"])

print("Campaign rows after dropping duplicates:", df_campaigns.shape)

Campaign rows before dropping duplicates: (337, 4)
Campaign rows after dropping duplicates: (337, 4)


In [ ]:
# merge into exploded df
df_model_base = df_exploded.merge(
    df_campaigns,
    on="mailing_id",
    how="left"
)

In [38]:
# validate merge
print("Before merge:", df_exploded.shape)
print("After merge:", df_model_base.shape)

print("\nUnique campaigns in exploded data:", df_exploded["mailing_id"].nunique())
print("Unique campaigns in campaign metadata:", df_campaigns["mailing_id"].nunique())

print("\nMissing mailing_info rows:", df_model_base["mailing_info"].isna().sum())
print("Missing subject_line rows:", df_model_base["subject_line"].isna().sum())
print("Missing preheader rows:", df_model_base["preheader"].isna().sum())

print("\nCampaigns without metadata:")
missing_metadata_campaigns = (
    df_model_base.loc[df_model_base["mailing_info"].isna(), "mailing_id"]
    .dropna()
    .unique()
)

print(len(missing_metadata_campaigns))
print(missing_metadata_campaigns[:20])

Before merge: (1156304, 21)
After merge: (1156304, 24)

Unique campaigns in exploded data: 274
Unique campaigns in campaign metadata: 337

Missing mailing_info rows: 28850
Missing subject_line rows: 28850
Missing preheader rows: 34359

Campaigns without metadata:
3
<IntegerArray>
[1084, 1207, 1083]
Length: 3, dtype: Int64


In [39]:
# safety checks
assert len(df_model_base) == len(df_exploded), "Merge changed number of rows."
assert df_model_base["user_id"].nunique() == df_exploded["user_id"].nunique(), "User count changed after merge."
assert df_model_base["mailing_id"].nunique() == df_exploded["mailing_id"].nunique(), "Campaign count changed after merge."

print("\nCampaign metadata merge checks passed.")


Campaign metadata merge checks passed.


In [42]:
print(df_model_base.shape)

print(
    df_model_base[
        ["user_id", "mailing_id", "open", "click"]
    ].head()
)

(1156304, 24)
                    user_id  mailing_id  open  click
0  5cb9a38136dd1336b9c528d1         668   0.0    0.0
1  5cb9a38136dd1336b9c528d1         692   0.0    0.0
2  5cb9a38136dd1336b9c528d1         714   0.0    0.0
3  5cb9a38136dd1336b9c528d1         733   0.0    0.0
4  5cb9a38136dd1336b9c528d1         761   0.0    0.0


### Remark: 3 campaigns without metadata

In [43]:
# create a flag
df_model_base["has_campaign_metadata"] = df_model_base["mailing_info"].notna()

In [44]:
display(
    df_model_base.groupby("has_campaign_metadata")[["open", "click"]]
    .mean()
    .assign(n_rows=df_model_base.groupby("has_campaign_metadata").size())
)

,open,click,n_rows
has_campaign_metadata,,,
False,0.367773,0.000544,28850
True,0.540546,0.001096,1127454


## 1.7 Create campaign topic features


In [51]:
# create campaign-topic table
campaign_topics = df_campaigns.copy()

campaign_topics["combined_info"] = (
    campaign_topics["mailing_info"].fillna("").astype(str) + " " +
    campaign_topics["subject_line"].fillna("").astype(str) + " " +
    campaign_topics["preheader"].fillna("").astype(str)
)

campaign_topics["campaign_topic"] = "Topic Unknown"

In [53]:
# apply topic rules
topic_rules = {
    "Automotive & Mobility": r"Mazda|Audi|Hyundai|Toyota|mobiliteit",
    "Media & Publishing": r"magazine|krant|film|media|De Telegraaf|Libelle|Donald|economie|VARAgids|politiek|landleven|wetenschap",
    "Finance & Investment": r"inkomen|pensioen|Roadway|capital|invest|beleggen|AOV|Tikkie|autoverzekering",
    "Energy & Sustainability": r"Engie|zonne|powerbank|airco",
    "Charity & Social Impact": r"goede doelen|hulp|Lepra|Oranje Fond|burendag",
    "Business": r"MKbasics|RANDSTAD|Regus|zaken|ziekteverzuim|groeien",
    "Food & Beverages": r"Nespresso|Uitgekookt",
    "Telecom & Technology": r"Odido",
    "Health & Wellbeing": r"Belife|mentaal|Pearle",
    "Lottery & Games": r"loterij|Stempunt|prijzenrad|lifepoints|e-bike|geluk",
    "Retail & Promotion": r"bespaartips|deals|Welke deal past bij jou deze week?"
}

for topic, pattern in topic_rules.items():
    mask = campaign_topics["combined_info"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
    campaign_topics.loc[mask, "campaign_topic"] = topic

In [54]:
# merge into campaign df
df_model_base = df_model_base.merge(
    campaign_topics[["mailing_id", "campaign_topic"]],
    on="mailing_id",
    how="left"
)

df_model_base["campaign_topic"] = df_model_base["campaign_topic"].fillna("Metadata Missing")

In [56]:
# validate
print("Campaign topic distribution by campaign:")
display(
    campaign_topics["campaign_topic"]
    .value_counts(dropna=False)
)

print("\nCampaign topic distribution by user-campaign rows:")
display(
    df_model_base["campaign_topic"]
    .value_counts(dropna=False)
)

print("\nMissing campaign topic rows:")
print((df_model_base["campaign_topic"] == "Metadata Missing").sum())

Campaign topic distribution by campaign:


campaign_topic
Automotive & Mobility      78
Media & Publishing         77
Business                   38
Charity & Social Impact    32
Finance & Investment       22
Lottery & Games            21
Food & Beverages           20
Topic Unknown              12
Health & Wellbeing         12
Energy & Sustainability    12
Retail & Promotion         10
Telecom & Technology        3
Name: count, dtype: int64


Campaign topic distribution by user-campaign rows:


campaign_topic
Media & Publishing         479712
Automotive & Mobility      257994
Lottery & Games            128094
Charity & Social Impact     98333
Energy & Sustainability     58188
Metadata Missing            28850
Health & Wellbeing          25822
Retail & Promotion          25563
Business                    21087
Finance & Investment        15722
Food & Beverages            15275
Telecom & Technology         1664
Name: count, dtype: int64


Missing campaign topic rows:
28850


## 1.8 Create recipient-campaign relevance features

In [71]:
# map each raw interest label to a broader topic
interest_to_topic_map = {
    # automotive
    'auto': 'Automotive & Mobility',
    'cx80': 'Automotive & Mobility',
    'auto inruilen': 'Automotive & Mobility',
    'mazda': 'Automotive & Mobility',
    'elektrisch': 'Automotive & Mobility',
    'audi': 'Automotive & Mobility',
    'mazda6e': 'Automotive & Mobility',
    'dacia': 'Automotive & Mobility',
    'hybrid': 'Automotive & Mobility',
    'mercedes': 'Automotive & Mobility',
    'toyota': 'Automotive & Mobility',
    'lease': 'Automotive & Mobility',
    'cx60': 'Automotive & Mobility',
    'cx5': 'Automotive & Mobility',
    'mx30': 'Automotive & Mobility',
    'cx30': 'Automotive & Mobility',
    'hyundai': 'Automotive & Mobility',
    'bedrijfswagen': 'Automotive & Mobility',
    'porsche': 'Automotive & Mobility',

    # business
    'mkb': 'Business',
    'ondernemers': 'Business',
    'zakelijk rijden': 'Business',
    'hr': 'Business',
    'bedrijven': 'Business',
    'zzp': 'Business',

    # charity
    'goededoelen': 'Charity & Social Impact',

    # energy
    'energie': 'Energy & Sustainability',

    # finance
    'verzekering': 'Finance & Investment',
    'bank': 'Finance & Investment',
    'beleggen': 'Finance & Investment',
    'finance': 'Finance & Investment',
    'hypotheek': 'Finance & Investment',
    'centraal beheer': 'Finance & Investment',
    'belastingdienst': 'Finance & Investment',

    # food
    'bezorgmaaltijden': 'Food & Beverages',
    'horeca': 'Food & Beverages',

    # health
    'zorg': 'Health & Wellbeing',

    # lottery
    'kansspelen': 'Lottery & Games',
    'loterij': 'Lottery & Games',
    'winactie': 'Lottery & Games',

    # media
    'bladen': 'Media & Publishing',
    'kranten': 'Media & Publishing',
    'entertainment': 'Media & Publishing',
    'onderzoek': 'Media & Publishing',

    # retail
    'cadeau': 'Retail & Promotion',
    'acties': 'Retail & Promotion',
    'kortingen': 'Retail & Promotion',

    # tech
    'windows': 'Telecom & Technology',
    'it': 'Telecom & Technology',
    'telcom': 'Telecom & Technology',
    'ios': 'Telecom & Technology',
    'android': 'Telecom & Technology',
    'linux': 'Telecom & Technology',
    'chrome os': 'Telecom & Technology',
    'mac os': 'Telecom & Technology',
    'os x': 'Telecom & Technology'
}

In [72]:
def map_user_interest_topics(interests):
    if not isinstance(interests, list):
        return []

    mapped_topics = []

    for interest in interests:
        interest = str(interest).strip().lower()
        mapped_topics.append(interest_to_topic_map.get(interest, "Other"))

    return mapped_topics

In [73]:
df_model_base["mapped_interest_topics"] = df_model_base["interests"].apply(
    map_user_interest_topics
)

In [74]:
def check_interest_topic_match(row):
    mapped_topics = row["mapped_interest_topics"]

    if not isinstance(mapped_topics, list):
        return 0

    return int(row["campaign_topic"] in mapped_topics)

In [75]:
df_model_base["interest_topic_match"] = df_model_base.apply(
    check_interest_topic_match,
    axis=1
)

In [ ]:
# validate
display(df_model_base["interest_topic_match"].value_counts(normalize=True, dropna=False))

display(
    df_model_base.groupby("interest_topic_match")[["open", "click"]]
    .mean()
    .assign(n_rows=df_model_base.groupby("interest_topic_match").size())
)

interest_topic_match
1    0.902153
0    0.097847
Name: proportion, dtype: float64

,open,click,n_rows
interest_topic_match,,,
0,0.318618,0.000742,113141
1,0.554747,0.001118,1043163


In [ ]:
# compare match rate after excluding rows with missing campaign metadata
df_check = df_model_base[df_model_base["campaign_topic"] != "Metadata Missing"].copy()

print("Full data match rate:")
print(df_model_base["interest_topic_match"].value_counts(normalize=True, dropna=False))

print("\nExcluding Metadata Missing:")
print(df_check["interest_topic_match"].value_counts(normalize=True, dropna=False))

Full data match rate:
interest_topic_match
1    0.902153
0    0.097847
Name: proportion, dtype: float64

Excluding Metadata Missing:
interest_topic_match
1    0.925238
0    0.074762
Name: proportion, dtype: float64


In [79]:
print(df_model_base["campaign_topic"].unique())

['Media & Publishing' 'Energy & Sustainability' 'Lottery & Games'
 'Charity & Social Impact' 'Automotive & Mobility' 'Metadata Missing'
 'Retail & Promotion' 'Health & Wellbeing' 'Finance & Investment'
 'Business' 'Food & Beverages' 'Telecom & Technology']


## 1.9 Create text features

In [85]:
# Create working text columns
df_model_base["subject_line_clean"] = (
    df_model_base["subject_line"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_model_base["preheader_clean"] = (
    df_model_base["preheader"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Missing metadata/text flags
df_model_base["subject_missing"] = df_model_base["subject_line"].isna()
df_model_base["preheader_missing"] = df_model_base["preheader"].isna()

# Length features
df_model_base["subject_length"] = df_model_base["subject_line_clean"].str.len()
df_model_base["preheader_length"] = df_model_base["preheader_clean"].str.len()

In [86]:
# Subject length groups from EDA
df_model_base["subject_length_group"] = pd.cut(
    df_model_base["subject_length"],
    bins=[0, 35, 45, 55, 100],
    labels=["short", "medium", "long", "very_long"],
    include_lowest=True
)

# Preheader length groups from EDA
df_model_base["preheader_length_group"] = pd.cut(
    df_model_base["preheader_length"],
    bins=[0, 45, 55, 65, 100],
    labels=["short", "medium", "long", "very_long"],
    include_lowest=True
)

In [87]:
# Number features
df_model_base["subject_contains_number"] = (
    df_model_base["subject_line_clean"].str.contains(r"\d", regex=True, na=False)
)

df_model_base["preheader_contains_number"] = (
    df_model_base["preheader_clean"].str.contains(r"\d", regex=True, na=False)
)

In [91]:
# Promotion wording features from EDA
promotion_words = ['korting', 'gratis', 'cadeau', 'aanbieding', 'deal',
                   'voordeel', 'win', 'actie', 'bonus', 'besparen']

promotion_pattern = "|".join(promotion_words)

df_model_base["subject_contains_promotion"] = (
    df_model_base["subject_line_clean"]
    .str.lower()
    .str.contains(promotion_pattern, regex=True, na=False)
)

df_model_base["preheader_contains_promotion"] = (
    df_model_base["preheader_clean"]
    .str.lower()
    .str.contains(promotion_pattern, regex=True, na=False)
)

In [92]:
# Exclaimation mark feature
df_model_base["subject_contains_exclamation"] = (
    df_model_base["subject_line_clean"].str.contains("!", regex=False, na=False)
)

df_model_base["preheader_contains_exclamation"] = (
    df_model_base["preheader_clean"].str.contains("!", regex=False, na=False)
)

In [93]:
text_feature_cols = [
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation"
]

display(df_model_base[text_feature_cols].head())

for col in text_feature_cols:
    print(f"\n{col}")
    display(df_model_base[col].value_counts(dropna=False))

,subject_missing,subject_length,subject_length_group,subject_contains_number,subject_contains_promotion,subject_contains_exclamation,preheader_missing,preheader_length,preheader_length_group,preheader_contains_number,preheader_contains_promotion,preheader_contains_exclamation
0,False,43,medium,True,True,False,False,48,medium,False,False,False
1,False,51,long,True,False,False,False,58,long,False,False,False
2,False,49,long,True,True,False,False,44,short,False,False,False
3,False,45,medium,False,False,False,False,65,long,False,True,False
4,False,35,short,False,False,True,False,41,short,True,True,False



subject_missing


subject_missing
False    1127454
True       28850
Name: count, dtype: int64


subject_length


subject_length
39    79296
43    75540
36    68139
51    67735
48    67457
47    65668
45    61383
44    58657
40    58461
49    56555
42    55367
46    53292
32    43151
58    37865
34    35678
0     28850
54    26913
59    25484
41    23366
57    23155
50    14791
38    14094
33    13663
62    11670
28    11547
35    10824
53    10756
75    10386
37     8162
56     6196
26     5464
93     4446
65     4106
63     3831
72     2941
31     2021
64     2009
74     1828
69     1458
55     1020
30      914
23      816
81      707
61      360
52      282
Name: count, dtype: int64

...

preheader_contains_number
False    612231
True     544073
Name: count, dtype: int64


preheader_contains_promotion


preheader_contains_promotion
False    890418
True     265886
Name: count, dtype: int64


preheader_contains_exclamation


preheader_contains_exclamation
False    1117260
True       39044
Name: count, dtype: int64

## 1.10 Add send-date audit information

In [95]:
print("Schedule raw shape:", df_schedule_raw.shape)
print(df_schedule_raw.columns.tolist())
display(df_schedule_raw.head())

Schedule raw shape: (1400, 11)
['Unnamed: 0', 'Subjectline', 'Mail ID ', 'Dag', 'Datum', 'Klant / bureau', 'Campagnenaam', 'Time stamp ', 'openInterst / clickInterest ', 'Mail content ', 'Unnamed: 10']


,Unnamed: 0,Subjectline,Mail ID,Dag,Datum,Klant / bureau,Campagnenaam,Time stamp,openInterst / clickInterest,Mail content,Unnamed: 10
0,ntvx,NaN,1417,Donderdag,2026-04-23 00:00:00,CPX,Donald Duck + Strandpakket,-Akkoord,"bladen, cadeaux",NaN,NaN
1,NaN,Wat zit er in Donalds strandtas?,NaN,NaN,NaN,NaN,NaN,Subjectline: Wat zit er in Donalds strandtas?,NaN,https://mail.omg.nl/x/?S7Y1.J9ra2hiaPm.CMjMyU_...,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pre-header: Ontdek het pakket en lees de leuks...,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260422 - CPX - CPX Donald Duck + Strandpakket ...,NaN,NaN,NaN
4,ntvx,NaN,1418,Donderdag,2026-04-23 00:00:00,CRS,Donald Duck + Strandpakket,-Akkoord,"bladen, cadeaux",https://mail.omg.nl/x/?S7Y1.J9ra2hiaPm.CMjMyU_...,NaN


In [96]:
# helper functions
def extract_all_ids(series):
    ids = []

    for value in series.dropna():
        value = str(value)

        found = re.findall(r"\d+", value)

        for item in found:
            ids.append(int(item))

    return ids


def first_valid(series):
    valid = series.dropna()

    if len(valid) == 0:
        return pd.NA

    return valid.iloc[0]

In [98]:
# clean raw structure
df_schedule = df_schedule_raw.copy()
df_schedule.columns = df_schedule.columns.str.strip()

# remove week separator rows
df_schedule = df_schedule[
    ~df_schedule["Unnamed: 0"]
    .astype(str)
    .str.contains("WK", case=False, na=False)
].copy()

In [99]:
# detect mailing blocks
is_ntvx = (
    df_schedule["Unnamed: 0"]
    .astype(str)
    .str.lower()
    .eq("ntvx")
)

is_header_like = (
    df_schedule["Mail ID"].astype(str).str.lower().eq("x")
    & df_schedule["Dag"].notna()
    & df_schedule["Datum"].notna()
    & df_schedule["Campagnenaam"].notna()
)

df_schedule["is_new_mailing"] = is_ntvx | is_header_like

df_schedule["block_id"] = df_schedule["is_new_mailing"].cumsum()

# remove anything before first detected mailing block
df_schedule = df_schedule[df_schedule["block_id"] > 0].copy()

In [100]:
# collapse each block
schedule_blocks = (
    df_schedule
    .groupby("block_id")
    .agg({
        "Mail ID": extract_all_ids,
        "Dag": first_valid,
        "Datum": first_valid,
        "Klant / bureau": first_valid,
        "Campagnenaam": first_valid,
        "Subjectline": first_valid,
        "Time stamp": lambda x: " | ".join(x.dropna().astype(str))
    })
    .reset_index()
)

In [101]:
# classify id status
schedule_blocks["n_mail_ids"] = schedule_blocks["Mail ID"].apply(len)

schedule_blocks["id_status"] = (
    schedule_blocks["n_mail_ids"]
    .map({
        0: "no_id",
        1: "single_id"
    })
    .fillna("multiple_ids")
)

schedule_blocks["mailing_id_clean"] = schedule_blocks["Mail ID"].apply(
    lambda x: x[0] if len(x) == 1 else pd.NA
)

In [102]:
# expand dataset
extra_schedule = schedule_blocks[
    schedule_blocks["id_status"].isin(["single_id", "multiple_ids"])
].copy()

extra_schedule = extra_schedule.explode("Mail ID").reset_index(drop=True)

extra_schedule = extra_schedule.rename(columns={
    "Mail ID": "mailing_id",
    "Dag": "send_day_of_week",
    "Datum": "send_date",
    "Time stamp": "timestamp_raw"
})

extra_schedule["mailing_id"] = extra_schedule["mailing_id"].astype(int)

extra_schedule = extra_schedule[
    [
        "mailing_id",
        "send_day_of_week",
        "send_date",
        "timestamp_raw",
        "block_id",
        "id_status"
    ]
].copy()

In [103]:
# keep only trusted records
dup_summary = (
    extra_schedule
    .groupby("mailing_id")
    .agg(
        n_rows=("mailing_id", "size"),
        n_dates=("send_date", "nunique"),
        dates=("send_date", lambda x: sorted(x.dropna().astype(str).unique())),
        statuses=("id_status", lambda x: sorted(x.dropna().astype(str).unique()))
    )
    .reset_index()
)

dup_summary["date_status"] = (
    dup_summary["n_dates"]
    .map({
        0: "no_date",
        1: "one_date"
    })
    .fillna("conflicting_dates")
)

trusted_ids = dup_summary.loc[
    dup_summary["date_status"] == "one_date",
    "mailing_id"
]

extra_schedule_trusted = extra_schedule[
    extra_schedule["mailing_id"].isin(trusted_ids)
].copy()

extra_schedule_trusted = (
    extra_schedule_trusted
    .sort_values(["mailing_id", "send_date"])
    .drop_duplicates(subset=["mailing_id"], keep="first")
    .copy()
)

In [104]:
# merge into primary df
df_model_base = df_model_base.merge(
    extra_schedule_trusted[
        [
            "mailing_id",
            "send_day_of_week",
            "send_date",
            "timestamp_raw"
        ]
    ],
    on="mailing_id",
    how="left"
)

df_model_base["date_trusted"] = df_model_base["send_date"].notna()

In [105]:
# validate
print("Trusted dated campaign IDs:", extra_schedule_trusted["mailing_id"].nunique())
print("Schedule rows trusted:", len(extra_schedule_trusted))

print("\nRow-level send_date coverage:")
print(df_model_base["send_date"].notna().mean())

print("\nCampaign-level send_date coverage:")
print(
    df_model_base
    .groupby("mailing_id")["send_date"]
    .first()
    .notna()
    .mean()
)

print("\nSend date missing rows:")
print(df_model_base["send_date"].isna().value_counts())

print("\nDate status distribution:")
display(dup_summary["date_status"].value_counts(dropna=False))

Trusted dated campaign IDs: 167
Schedule rows trusted: 167

Row-level send_date coverage:
0.673301311765764

Campaign-level send_date coverage:
0.551094890510949

Send date missing rows:
send_date
False    778541
True     377763
Name: count, dtype: int64

Date status distribution:


date_status
one_date             167
conflicting_dates     40
Name: count, dtype: int64

## 1.11 Create historical engagement features

In [106]:
# sort by user and campaign order
df_model_base = df_model_base.sort_values(
    ["user_id", "mailing_id"]
).copy()

In [107]:
# create prior cumulative counts
df_model_base["prior_open_count"] = (
    df_model_base.groupby("user_id")["open"].cumsum() - df_model_base["open"]
)

df_model_base["prior_click_count"] = (
    df_model_base.groupby("user_id")["click"].cumsum() - df_model_base["click"]
)

df_model_base["prior_email_count"] = (
    df_model_base.groupby("user_id").cumcount()
)

In [108]:
# create historical rates
df_model_base["historical_open_rate"] = (
    df_model_base["prior_open_count"] / df_model_base["prior_email_count"]
)

df_model_base["historical_click_rate"] = (
    df_model_base["prior_click_count"] / df_model_base["prior_email_count"]
)

In [109]:
# buckets for open and email frequency
df_model_base["historical_open_bucket"] = pd.cut(
    df_model_base["historical_open_rate"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["very_low", "low", "moderate", "high", "very_high"],
    include_lowest=True
)

df_model_base["prior_email_frequency_bucket"] = pd.cut(
    df_model_base["prior_email_count"],
    bins=[0, 5, 15, 30, 50, 120],
    labels=["very_little", "little", "moderate", "heavy", "very_high"],
    include_lowest=True
)

In [110]:
# prior click indicator
df_model_base["had_prior_click"] = df_model_base["historical_click_rate"] > 0

In [111]:
# first email indicator
df_model_base["is_first_email"] = (
    df_model_base["prior_email_count"] == 0
)

In [112]:
# validation
print("Prior email count summary:")
display(df_model_base["prior_email_count"].describe())

print("\nHistorical open rate summary:")
display(df_model_base["historical_open_rate"].describe())

print("\nHistorical click rate summary:")
display(df_model_base["historical_click_rate"].describe())

print("\nHistorical open bucket distribution:")
display(df_model_base["historical_open_bucket"].value_counts(dropna=False))

print("\nPrior email frequency bucket distribution:")
display(df_model_base["prior_email_frequency_bucket"].value_counts(dropna=False))

print("\nHad prior click distribution:")
display(df_model_base["had_prior_click"].value_counts(dropna=False))

print("\nFirst email rows:")
print(df_model_base["is_first_email"].sum())

Prior email count summary:


count    1.156304e+06
mean     3.224586e+01
std      2.142746e+01
min      0.000000e+00
25%      1.400000e+01
50%      3.000000e+01
75%      4.800000e+01
max      1.190000e+02
Name: prior_email_count, dtype: float64


Historical open rate summary:


count    1.039581e+06
mean     5.558329e-01
std      3.447998e-01
min      0.000000e+00
25%      2.352941e-01
50%      6.000000e-01
75%      8.888889e-01
max      1.000000e+00
Name: historical_open_rate, dtype: float64


Historical click rate summary:


...


Prior email frequency bucket distribution:


prior_email_frequency_bucket
heavy          325864
moderate       266645
very_high      247494
little         196301
very_little    120000
Name: count, dtype: int64


Had prior click distribution:


had_prior_click
False    1147533
True        8771
Name: count, dtype: int64


First email rows:
20000


## 1.12 Freeze modelling dataframe

In [117]:
# target-specific views
df_open_model_v1 = df_model_base[df_model_base["open"].notna()].copy()
df_click_model_v1 = df_model_base[df_model_base["click"].notna()].copy()

print("Full candidate dataset:", df_model_base.shape)
print("Open model dataset:", df_open_model_v1.shape)
print("Click model dataset:", df_click_model_v1.shape)

Full candidate dataset: (1156304, 59)
Open model dataset: (1057544, 59)
Click model dataset: (1104242, 59)


In [118]:
# target distributions
print("Open target distribution:")
display(df_open_model_v1["open"].value_counts(normalize=True, dropna=False))

print("Click target distribution:")
display(df_click_model_v1["click"].value_counts(normalize=True, dropna=False))

Open target distribution:


open
1.0    0.53618
0.0    0.46382
Name: proportion, dtype: float64

Click target distribution:


click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

In [119]:
id_cols = [
    "user_id",
    "mailing_id"
]

target_cols = [
    "open",
    "click"
]

audit_cols = [
    "mailing_info",
    "subject_line",
    "preheader",
    "send_date",
    "send_day_of_week",
    "date_trusted",
    "has_campaign_metadata"
]

recipient_feature_cols = [
    "gender",
    "age_clean",
    "age_group",
    "n_interests",
    "interest_topic_match"
]

campaign_feature_cols = [
    "campaign_topic",
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation"
]

historical_feature_cols = [
    "prior_email_count",
    "prior_open_count",
    "prior_click_count",
    "historical_open_rate",
    "historical_click_rate",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

In [120]:
freeze_cols = (
    id_cols
    + target_cols
    + audit_cols
    + recipient_feature_cols
    + campaign_feature_cols
    + historical_feature_cols
)

df_model_candidates_v1 = df_model_base[freeze_cols].copy()

In [121]:
print("Frozen modelling dataset shape:", df_model_candidates_v1.shape)
print("Number of users:", df_model_candidates_v1["user_id"].nunique())
print("Number of campaigns:", df_model_candidates_v1["mailing_id"].nunique())

print("\nOpen missing rate:")
print(df_model_candidates_v1["open"].isna().mean())

print("\nClick missing rate:")
print(df_model_candidates_v1["click"].isna().mean())

display(df_model_candidates_v1.head())

Frozen modelling dataset shape: (1156304, 38)
Number of users: 20000
Number of campaigns: 274

Open missing rate:
0.08541006517317246

Click missing rate:
0.045024491829138355


,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,has_campaign_metadata,gender,age_clean,age_group,n_interests,interest_topic_match,campaign_topic,subject_missing,subject_length,subject_length_group,subject_contains_number,subject_contains_promotion,subject_contains_exclamation,preheader_missing,preheader_length,preheader_length_group,preheader_contains_number,preheader_contains_promotion,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
349935,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25 00:00:00,Vrijdag,True,1,male,NaN,NaN,17,1,Media & Publishing,False,43,medium,True,True,False,False,48,medium,False,False,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
349936,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,False,False,False,63,long,False,True,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349937,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,True,False,False,44,short,False,False,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349938,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14 00:00:00,Woensdag,True,1,male,NaN,NaN,17,1,Lottery & Games,False,45,medium,False,False,False,False,65,long,False,True,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349939,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,35,short,False,False,True,False,41,short,True,True,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


## 1.13 Build feature registry

In [122]:
df_open_model_v1 = df_model_candidates_v1[
    df_model_candidates_v1["open"].notna()
].copy()

df_click_model_v1 = df_model_candidates_v1[
    df_model_candidates_v1["click"].notna()
].copy()

In [123]:
print(df_open_model_v1.shape)
print(df_click_model_v1.shape)

display(
    df_open_model_v1["open"]
    .value_counts(normalize=True)
)

display(
    df_click_model_v1["click"]
    .value_counts(normalize=True)
)

(1057544, 38)
(1104242, 38)


open
1.0    0.53618
0.0    0.46382
Name: proportion, dtype: float64

click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

## 1.14 Save frozen outputs

In [128]:
df_model_candidates_v1.to_parquet(
    PROCESSED_DATA_DIR / "df_model_candidates_v1.parquet",
    index=False
)

df_open_model_v1.to_parquet(
    PROCESSED_DATA_DIR / "df_open_model_v1.parquet",
    index=False
)

df_click_model_v1.to_parquet(
    PROCESSED_DATA_DIR / "df_click_model_v1.parquet",
    index=False
)

In [129]:
print(PROCESSED_DATA_DIR)
print(PROCESSED_DATA_DIR.exists())

/Users/khanhhien/Documents/Thesis/Data/Processed Data
True


## 1.15 Dataset checks

In [130]:
# shape
print(df_model_candidates_v1.shape)
print(df_open_model_v1.shape)
print(df_click_model_v1.shape)

(1156304, 38)
(1057544, 38)
(1104242, 38)


In [132]:
# target distributions
print(df_open_model_v1["open"].value_counts(normalize=True))

print(df_click_model_v1["click"].value_counts(normalize=True))

open
1.0    0.53618
0.0    0.46382
Name: proportion, dtype: float64
click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64


In [133]:
# missing values
missing_check = (
    df_model_candidates_v1
    .isna()
    .mean()
    .sort_values(ascending=False)
)

display(missing_check.head(20))

age_group                         0.743566
age_clean                         0.743566
send_date                         0.326699
send_day_of_week                  0.326699
historical_open_bucket            0.100945
historical_open_rate              0.100945
prior_open_count                  0.085410
open                              0.085410
historical_click_rate             0.061640
click                             0.045024
prior_click_count                 0.045024
preheader                         0.029715
subject_line                      0.024950
mailing_info                      0.024950
prior_email_count                 0.000000
preheader_contains_promotion      0.000000
preheader_contains_exclamation    0.000000
preheader_contains_number         0.000000
user_id                           0.000000
preheader_length                  0.000000
dtype: float64

In [134]:
# leakage variables check
print(sorted(df_model_candidates_v1.columns))

['age_clean', 'age_group', 'campaign_topic', 'click', 'date_trusted', 'gender', 'had_prior_click', 'has_campaign_metadata', 'historical_click_rate', 'historical_open_bucket', 'historical_open_rate', 'interest_topic_match', 'is_first_email', 'mailing_id', 'mailing_info', 'n_interests', 'open', 'preheader', 'preheader_contains_exclamation', 'preheader_contains_number', 'preheader_contains_promotion', 'preheader_length', 'preheader_length_group', 'preheader_missing', 'prior_click_count', 'prior_email_count', 'prior_email_frequency_bucket', 'prior_open_count', 'send_date', 'send_day_of_week', 'subject_contains_exclamation', 'subject_contains_number', 'subject_contains_promotion', 'subject_length', 'subject_length_group', 'subject_line', 'subject_missing', 'user_id']


In [135]:
# first few rows
display(df_model_candidates_v1.head())

,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,has_campaign_metadata,gender,age_clean,age_group,n_interests,interest_topic_match,campaign_topic,subject_missing,subject_length,subject_length_group,subject_contains_number,subject_contains_promotion,subject_contains_exclamation,preheader_missing,preheader_length,preheader_length_group,preheader_contains_number,preheader_contains_promotion,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
349935,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25 00:00:00,Vrijdag,True,1,male,NaN,NaN,17,1,Media & Publishing,False,43,medium,True,True,False,False,48,medium,False,False,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
349936,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,False,False,False,63,long,False,True,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349937,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,49,long,True,True,False,False,44,short,False,False,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349938,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14 00:00:00,Woensdag,True,1,male,NaN,NaN,17,1,Lottery & Games,False,45,medium,False,False,False,False,65,long,False,True,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
349939,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04 00:00:00,Woensdag,True,1,male,NaN,NaN,17,0,Energy & Sustainability,False,35,short,False,False,True,False,41,short,True,True,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


In [136]:
# duplicate rows check
df_model_candidates_v1.duplicated().sum()

0

In [137]:
# unique users and campaigns check
print(df_model_candidates_v1["user_id"].nunique())
print(df_model_candidates_v1["mailing_id"].nunique())

20000
274
